In [ ]:
import pandas as pd
import os
import matplotlib.pyplot as plt
import glob

print("Libraries imported successfully.")

In [ ]:
results_dir = 'resultados'
experiment_data = []

# Find all stat files recursively in the results directory
stat_files = glob.glob(os.path.join(results_dir, '**', '*_stats.txt'), recursive=True)

for file_path in stat_files:
    data = {}
    with open(file_path, 'r') as f:
        for line in f:
            if ':' in line:
                key, value = line.split(':', 1)
                data[key.strip()] = value.strip()
    
    # Extract instance name and lambda from the file path for clarity
    parts = file_path.split(os.sep)
    instance_info = parts[-2] # e.g., 'run1_P1_k5_lambda_0.0'
    
    data['instance_full'] = instance_info
    data['instance_base'] = instance_info.split('_lambda_')[0]
    data['lambda'] = float(instance_info.split('_lambda_')[1])
    
    experiment_data.append(data)

# Create a clean DataFrame
df_results = pd.DataFrame(experiment_data)

# Convert columns to numeric types for plotting and analysis
df_results['Minimum Expected Imbalance'] = pd.to_numeric(df_results['Minimum Expected Imbalance'])
df_results['Number of Clusters Found'] = pd.to_numeric(df_results['Number of Clusters Found'])
df_results['Solver Execution Time (s)'] = pd.to_numeric(df_results['Solver Execution Time (s)'])

print(f"Loaded {len(df_results)} experiment results.")
df_results.sort_values(by=['instance_base', 'lambda'], inplace=True)
df_results.head()

In [ ]:
# Select a specific instance to analyze, for example, P1
instance_to_plot = 'run1_P1_k5'
df_instance = df_results[df_results['instance_base'] == instance_to_plot].copy()

# --- Plotting the Pareto Front ---
plt.style.use('seaborn-v0_8-whitegrid')
fig, ax = plt.subplots(figsize=(10, 7))

# Plot each point (solution)
ax.plot(
    df_instance['Number of Clusters Found'], 
    df_instance['Minimum Expected Imbalance'], 
    marker='o', 
    linestyle='--',
    color='b',
    label='Soluções Encontradas'
)

# Add labels for each lambda value
for i, row in df_instance.iterrows():
    ax.text(
        row['Number of Clusters Found'] + 0.1, 
        row['Minimum Expected Imbalance'], 
        f"λ={row['lambda']}", 
        fontsize=12
    )

ax.set_xlabel('Número de Clusters (k) - (Objetivo $f_2$)', fontsize=14)
ax.set_ylabel('Desequilíbrio Esperado - (Objetivo $f_1$)', fontsize=14)
ax.set_title(f'Fronteira de Pareto para a Instância: {instance_to_plot}', fontsize=16, pad=20)
ax.legend()
ax.grid(True)

# Save the figure to be used in the dissertation
output_fig_path = os.path.join('resultados', f'pareto_{instance_to_plot}.png')
plt.savefig(output_fig_path, dpi=300)
print(f"Pareto front plot saved to '{output_fig_path}'")

plt.show()

In [ ]:
# Filter for the same instance
df_instance = df_results[df_results['instance_base'] == instance_to_plot]

# Solution with minimum number of clusters (when lambda is 0.0)
min_k_solution = df_instance[df_instance['lambda'] == 0.0]
print("--- Solução com Foco em SIMPLICIDADE (lambda=0.0) ---")
print(min_k_solution[['Number of Clusters Found', 'Minimum Expected Imbalance', 'instance_full']])
print("-" * 50)


# Solution with minimum disagreement (when lambda is 1.0)
min_disagreement_solution = df_instance[df_instance['lambda'] == 1.0]
print("\n--- Solução com Foco em QUALIDADE (lambda=1.0) ---")
print(min_disagreement_solution[['Number of Clusters Found', 'Minimum Expected Imbalance', 'instance_full']])
print("-" * 50)

# A balanced/trade-off solution (e.g., lambda=0.5)
balanced_solution = df_instance[df_instance['lambda'] == 0.5]
print("\n--- Solução de COMPROMISSO (lambda=0.5) ---")
print(balanced_solution[['Number of Clusters Found', 'Minimum Expected Imbalance', 'instance_full']])
print("-" * 50)